# 05 — From a complete PDF to an evidence-grounded answer

**Read this notebook for the integrated system and its completion status.** It runs entirely from verified public aggregates: no login, private PDF, model download, or paid GPU is needed to inspect it. A missing end-to-end result is shown as **not yet measured**, never filled in with oracle scores or synthetic tests.

The engineering question is specific: can the selected 9B reader answer from five pages selected by multilingual BM25, instead of being handed the correct evidence? The reader and semantic judge remain pinned. This is a fixed inference experiment, **not training or fine-tuning**.

## 1. Establish the evidence boundary

In [1]:
import json
from pathlib import Path

from IPython.display import HTML, display

from lava.evaluation.reporting import load_report
from lava.evaluation.system import load_summary
from lava.evaluation.walkthrough import TABLE_STYLE, render_table
from lava.notebook_support import find_repo_root
from lava.readers.runtime_logging import RuntimeEventLogger
from lava.retrieval.pipeline import load_public_report
from lava.retrieval.reporting import recall_chart

ROOT = find_repo_root(Path.cwd())
logger = RuntimeEventLogger("notebook.system")
with logger.stage("01_verify_evidence", heartbeat_seconds=15):
    readers = load_report(ROOT)
    baseline = next(
        row for row in readers["current_models"] if row["model_key"] == "qwen35_9b_fused_direct"
    )
    assert baseline["complete"] and baseline["semantic_summary"]["contract_current"]
    oracle = baseline["semantic_summary"]["metrics"]
    retrieval = load_public_report(ROOT)
    result = load_summary(ROOT)
    status = result["status"] if result else "End-to-end inference not yet measured"
    display(
        HTML(
            TABLE_STYLE
            + render_table(
                [
                    {"Evidence": "Current end-to-end status", "Observation": status},
                    {
                        "Evidence": "Oracle baseline",
                        "Observation": f"{oracle['question_micro']['overall']:.2%} local LAVA; correct pages supplied",
                    },
                    {
                        "Evidence": "Labeled pilot",
                        "Observation": "16 previously examined questions / 5 PDFs / 15 Japanese + 1 Vietnamese",
                    },
                    {
                        "Evidence": "Claim boundary",
                        "Observation": "Training diagnostic; no hidden-test, organizer-server or state-of-the-art claim",
                    },
                ],
                caption="Measured evidence, not promised performance",
            )
        )
    )

{"component": "notebook.system", "elapsed_seconds": 0.0, "event": "01_verify_evidence.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T03:56:59.041+00:00"}


Evidence,Observation
Current end-to-end status,End-to-end inference not yet measured
Oracle baseline,87.02% local LAVA; correct pages supplied
Labeled pilot,16 previously examined questions / 5 PDFs / 15 Japanese + 1 Vietnamese
Claim boundary,"Training diagnostic; no hidden-test, organizer-server or state-of-the-art claim"


{"component": "notebook.system", "elapsed_seconds": 0.377, "event": "01_verify_evidence.completed", "level": "INFO", "stage_elapsed_seconds": 0.377, "timestamp_utc": "2026-09-07T03:56:59.418+00:00"}


## 2. Design useful inputs, not more features indiscriminately

For this document-AI task, feature engineering means **preserving the information the answer depends on**. Native text captures searchable words and numbers; page images retain tables, charts, and spatial relationships; explicit physical page numbers support auditable citations. More context can also introduce distractors and consume memory.

The frozen retrieval representation uses Unicode NFKC normalization, words, and character bigrams/trigrams. This lets Japanese match without whitespace tokenization while retaining Vietnamese diacritics. Frequencies and page-length normalization are computed within the queried PDF. Gold answers and gold page labels never enter the reader request.

The five-page budget was fixed for this final pilot **after examining the earlier retrieval diagnostics**; it is a development choice, not a held-out selection. Ten pages retrieved all known evidence but have not been demonstrated to improve answer quality or latency. We do not equate retrieval recall with answer accuracy.

In [2]:
with logger.stage("02_inspect_input_tradeoff", heartbeat_seconds=15):
    display(HTML(recall_chart(retrieval, "all_evidence_at_k")))
    display(
        HTML(
            render_table(
                [
                    {
                        "Input decision": "PDF text + rendered image",
                        "Reason": "Lexical matching plus visual tables/layout; no destructive text-only assumption",
                    },
                    {
                        "Input decision": "Five ranked pages, shown in physical order",
                        "Reason": "Bounded context; stable page identities instead of renumbered citations",
                    },
                    {
                        "Input decision": "Pinned source and model hashes",
                        "Reason": "Detect changed inputs; reuse only compatible answers",
                    },
                    {
                        "Input decision": "Labels separated from ReaderInput",
                        "Reason": "Reference answers and pages become visible only in the evaluator",
                    },
                ],
                caption="Information-preserving input design",
            )
        )
    )

{"component": "notebook.system", "elapsed_seconds": 0.384, "event": "02_inspect_input_tradeoff.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T03:56:59.424+00:00"}


Input decision,Reason
PDF text + rendered image,Lexical matching plus visual tables/layout; no destructive text-only assumption
"Five ranked pages, shown in physical order",Bounded context; stable page identities instead of renumbered citations
Pinned source and model hashes,Detect changed inputs; reuse only compatible answers
Labels separated from ReaderInput,Reference answers and pages become visible only in the evaluator


{"component": "notebook.system", "elapsed_seconds": 0.386, "event": "02_inspect_input_tradeoff.completed", "level": "INFO", "stage_elapsed_seconds": 0.003, "timestamp_utc": "2026-09-07T03:56:59.427+00:00"}


## 3. Follow the actual execution path

Frozen source versions → reuse full-PDF text extraction → reuse label-blind BM25 rankings → render only selected unique pages → construct label-free `ReaderInput` → invoke the same pinned 9B reader → independently parse the exact generation → persist and read back each answer → evaluate with the unchanged semantic judge.

The original oracle experiment retains its strict gold-page-alignment contract. The new input type has no reference answer or gold evidence fields. Output citations must be members of the **supplied physical page set**, not the reference set. Invalid generations remain failures in the denominator.

A successful AWS job status is necessary but insufficient: complete question coverage, checksums, re-parsed outputs, judge controls, and public report integrity must also pass.

In [3]:
with logger.stage("03_inspect_frozen_execution", heartbeat_seconds=15):
    config = json.loads((ROOT / "configs/system_evaluation.json").read_text())
    display(
        HTML(
            render_table(
                [
                    {
                        "Contract": "Reader",
                        "Value": "Qwen3.5-9B / bfloat16 / direct deterministic decoding",
                    },
                    {"Contract": "Page budget", "Value": config["page_budget"]},
                    {
                        "Contract": "GPU attempt",
                        "Value": "One ml.g6e.2xlarge; 1,800-second server runtime cap; no endpoint",
                    },
                    {
                        "Contract": "Resume",
                        "Value": "Reattach to a deterministic job name; skip verified answer checkpoints",
                    },
                    {
                        "Contract": "New paid retry",
                        "Value": "Separate explicit approval; never automatic",
                    },
                ],
                caption="A bounded experiment, not an open-ended search",
            )
        )
    )

{"component": "notebook.system", "elapsed_seconds": 0.392, "event": "03_inspect_frozen_execution.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T03:56:59.433+00:00"}


Contract,Value
Reader,Qwen3.5-9B / bfloat16 / direct deterministic decoding
Page budget,5
GPU attempt,"One ml.g6e.2xlarge; 1,800-second server runtime cap; no endpoint"
Resume,Reattach to a deterministic job name; skip verified answer checkpoints
New paid retry,Separate explicit approval; never automatic


{"component": "notebook.system", "elapsed_seconds": 0.394, "event": "03_inspect_frozen_execution.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-07T03:56:59.435+00:00"}


## 4. Compare the official-formula components

The published LAVA formula averages semantic answer credit and evidence-page F1 per question. Unordered list answers use optimal semantic matching; ordered lists use semantic longest-common-subsequence credit. Our pinned local Gemma judge implements this evaluation structure, but the organizer's exact prompt/runtime is not published, so these are **local published-formula scores**, not official leaderboard scores.

The comparison below uses actual predicted evidence pages. Supplying the reference pages during scoring would incorrectly inflate the complete-system score. The oracle-to-retrieved difference includes missed evidence, distractor pages, and context-volume changes; it is not a pure causal estimate of retrieval alone.

In [4]:
def percent(value):
    return "Not yet measured" if value is None else f"{value:.2%}"


with logger.stage("04_compare_quality", heartbeat_seconds=15):
    current = result["metrics"]["question_micro"] if result else {}
    rows = [
        {
            "Metric": label,
            "Oracle pages": percent(oracle["question_micro"][key]),
            "Retrieved pages": percent(current.get(key)),
            "Change (points)": f"{100 * (current[key] - oracle['question_micro'][key]):+.2f}"
            if key in current
            else "Not yet measured",
        }
        for key, label in (
            ("answer", "Semantic answer credit"),
            ("grounding", "Evidence-page F1"),
            ("overall", "Local LAVA overall"),
        )
    ]
    display(HTML(render_table(rows, caption="Same 9B configuration and semantic judge")))
    if result:
        display(
            HTML(
                render_table(
                    [
                        {
                            "Weighting": "Question average",
                            "Oracle": percent(oracle["question_micro"]["overall"]),
                            "Retrieved": percent(current["overall"]),
                        },
                        {
                            "Weighting": "Equal-document average",
                            "Oracle": percent(oracle["document_macro"]["overall"]),
                            "Retrieved": percent(result["metrics"]["document_macro"]["overall"]),
                        },
                    ],
                    caption="Do not let PDFs with more questions hide inconsistent behavior",
                )
            )
        )

{"component": "notebook.system", "elapsed_seconds": 0.401, "event": "04_compare_quality.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T03:56:59.442+00:00"}


Metric,Oracle pages,Retrieved pages,Change (points)
Semantic answer credit,80.15%,Not yet measured,Not yet measured
Evidence-page F1,93.90%,Not yet measured,Not yet measured
Local LAVA overall,87.02%,Not yet measured,Not yet measured


{"component": "notebook.system", "elapsed_seconds": 0.404, "event": "04_compare_quality.completed", "level": "INFO", "stage_elapsed_seconds": 0.003, "timestamp_utc": "2026-09-07T03:56:59.445+00:00"}


## 5. Diagnose the errors before proposing another model

Each question is assigned one diagnostic category in this order: invalid response; missing required evidence; incomplete/incorrect answer despite complete evidence; incomplete citation; full credit. These categories are operational triage, not proof of causation. A missing page and an answer error can coexist.

Report every PDF and keep the single Vietnamese question visible as a sample-size limitation, not a language benchmark. The document-level paired interval is exploratory: only five PDFs were available, and all were previously examined during development.

In [5]:
with logger.stage("05_examine_failures", heartbeat_seconds=15):
    if result:
        display(
            HTML(
                render_table(
                    [
                        {"Failure category": category.replace("_", " "), "Questions": count}
                        for category, count in result["diagnostics"]["failure_counts"].items()
                    ],
                    caption="All 16 questions remain accounted for",
                )
            )
        )
        display(
            HTML(
                render_table(
                    [
                        {
                            "PDF": name,
                            "Questions": sum(
                                row["document"] == name for row in result["per_question"]
                            ),
                            "Oracle LAVA": percent(oracle["by_document"][name]["overall"]),
                            "Retrieved LAVA": percent(values["overall"]),
                            "Change (points)": f"{100 * (values['overall'] - oracle['by_document'][name]['overall']):+.2f}",
                        }
                        for name, values in result["metrics"]["by_document"].items()
                    ],
                    caption="Paired document consistency",
                )
            )
        )
        display(
            HTML(
                render_table(
                    [
                        {
                            "Question": row["question"],
                            "PDF": row["document"],
                            "Answer credit": percent(row["answer_score"]),
                            "Evidence F1": percent(row["evidence_f1"]),
                            "Diagnosis": row["failure_category"].replace("_", " "),
                        }
                        for row in result["per_question"]
                    ],
                    caption="Sanitized question diagnostics; no raw source data",
                )
            )
        )
    else:
        display(
            HTML(
                render_table(
                    [
                        {
                            "Analysis": "End-to-end error taxonomy",
                            "Result": "Awaiting actual retrieved-page generations",
                        },
                        {
                            "Analysis": "Known retrieval limitation",
                            "Result": "At k=5, complete evidence is missing for 2 of 16 questions",
                        },
                    ],
                    caption="What can and cannot be concluded now",
                )
            )
        )

{"component": "notebook.system", "elapsed_seconds": 0.412, "event": "05_examine_failures.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T03:56:59.453+00:00"}


Analysis,Result
End-to-end error taxonomy,Awaiting actual retrieved-page generations
Known retrieval limitation,"At k=5, complete evidence is missing for 2 of 16 questions"


{"component": "notebook.system", "elapsed_seconds": 0.414, "event": "05_examine_failures.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-07T03:56:59.455+00:00"}


## 6. Measure execution and preserve successful work

Generation latency, reader latency, and AWS billable time measure different things. Reader timing includes preprocessing and, for the first uncached question, model loading; it excludes retrieval, provisioning, and durable writes. Recovered answers can span multiple GPU attempts. Do not call these measurements full competition-runtime compliance.

Each operator invocation records UTC timestamps, stage/total elapsed time, and heartbeats. Exact generations, answer checkpoints, judge decisions, completed reports, and successful notebook publications are durable in private S3. Analysis reruns verify existing outputs instead of paying for new inference.

In [6]:
with logger.stage("06_inspect_runtime_and_recovery", heartbeat_seconds=15):
    if result:
        runtime = result["runtime"]
        display(
            HTML(
                render_table(
                    [
                        {"Measurement": name.replace("_", " "), "Value": value}
                        for name, value in runtime.items()
                    ],
                    caption="Measured reader runtime; scope is explicit",
                )
            )
        )
    else:
        display(
            HTML(
                render_table(
                    [
                        {
                            "Measurement": "Retrieved-page GPU runtime/memory",
                            "Value": "Not yet measured",
                        },
                        {
                            "Measurement": "Existing retrieval initial run",
                            "Value": f"{retrieval['first_run_seconds']:.3f} seconds",
                        },
                        {
                            "Measurement": "Resume guarantee",
                            "Value": "Verified completed checkpoints reused; an interrupted unsaved question may need repetition",
                        },
                    ],
                    caption="Measured timing versus unmeasured work",
                )
            )
        )
    prices = json.loads((ROOT / "reports/aws/training_prices.json").read_text())
    display(
        HTML(
            render_table(
                [
                    {
                        "Cost boundary": "Published price snapshot",
                        "Meaning": "See reports/aws/training_prices.json for dated Training rates",
                    },
                    {
                        "Cost boundary": "Explicit launch budget",
                        "Meaning": "Per-attempt compute estimate only; not a total AWS billing cap",
                    },
                    {
                        "Cost boundary": "Other charges",
                        "Meaning": "Existing Studio, storage, logs, data transfer and previous attempts are separate",
                    },
                ],
                caption="No misleading all-in cost claim",
            )
        )
    )

{"component": "notebook.system", "elapsed_seconds": 0.421, "event": "06_inspect_runtime_and_recovery.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T03:56:59.462+00:00"}


Measurement,Value
Retrieved-page GPU runtime/memory,Not yet measured
Existing retrieval initial run,8.375 seconds
Resume guarantee,Verified completed checkpoints reused; an interrupted unsaved question may need repetition


Cost boundary,Meaning
Published price snapshot,See reports/aws/training_prices.json for dated Training rates
Explicit launch budget,Per-attempt compute estimate only; not a total AWS billing cap
Other charges,"Existing Studio, storage, logs, data transfer and previous attempts are separate"


{"component": "notebook.system", "elapsed_seconds": 0.424, "event": "06_inspect_runtime_and_recovery.completed", "level": "INFO", "stage_elapsed_seconds": 0.003, "timestamp_utc": "2026-09-07T03:56:59.465+00:00"}


## 7. Close the portfolio milestone, not an endless experiment loop

**Acceptance criteria:** 16 real retrieved-page predictions; immutable provenance and complete coverage; unchanged judge controls passing; answer/evidence/local LAVA metrics plus failure analysis; executed notebooks; quality gate passing; documented results commit and reviewed PR. The observed score may be lower than the oracle score; that is a result to explain, not a reason to hide failures or silently change the experiment.

Run `make system-preview` to inspect the plan with no paid resource creation. In the existing configured SageMaker workspace, `make finish CHARGES=YES` explicitly authorizes the bounded GPU attempt and then scores, republishes all six notebooks, checks quality, and archives the exact successful publications. Repeat the same command after disconnection to reattach or reuse completed work. A failed/stopped GPU attempt requires a deliberately new `ATTEMPT` and `RETRY=YES`.

Kaggle submission is **optional and separate**: all 624 hidden-test predictions, container/runtime validation and eligibility remain necessary before claiming a submission. Viewing or completing this scoped research portfolio does not require a web application, another model sweep, or a leaderboard upload.

References: [official LAVA evaluation](https://lava-workshop.github.io/#evaluation), [BM25 implementation reference](https://lucene.apache.org/core/9_12_1/core/org/apache/lucene/search/similarities/BM25Similarity.html), [SageMaker stopping conditions](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_StoppingCondition.html).

In [7]:
with logger.stage("07_record_conclusion", heartbeat_seconds=15):
    if result:
        delta = result["question_mean_delta"]
        conclusion = (
            f"Retrieved-page local LAVA: {result['metrics']['question_micro']['overall']:.2%}; "
            f"change from oracle: {100 * delta:+.2f} percentage points. "
            "The integrated training pilot is measured. Retain its failures and limitations; publish this evidence rather than restarting model selection."
        )
    else:
        conclusion = (
            "Reader selection and retrieval diagnostics are measured; the integrated score is not. "
            "The next required action is the explicitly charge-gated 16-question GPU run, not another redesign."
        )
    display(HTML(render_table([{"Conclusion": conclusion}], caption="Final system evaluation")))
logger.emit("system.walkthrough.completed", measured=result is not None, question_count=16)

{"component": "notebook.system", "elapsed_seconds": 0.429, "event": "07_record_conclusion.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T03:56:59.470+00:00"}


Conclusion
"Reader selection and retrieval diagnostics are measured; the integrated score is not. The next required action is the explicitly charge-gated 16-question GPU run, not another redesign."


{"component": "notebook.system", "elapsed_seconds": 0.43, "event": "07_record_conclusion.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-07T03:56:59.471+00:00"}


{"component": "notebook.system", "elapsed_seconds": 0.431, "event": "system.walkthrough.completed", "level": "INFO", "measured": false, "question_count": 16, "timestamp_utc": "2026-09-07T03:56:59.472+00:00"}
